In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import joblib

# قراءة ملف الإكسل
df = pd.read_excel("Dalal_Sanaa_rent_1000_EN_unique.xlsx")

# عرض أول 5 صفوف للتأكد
df.head()


,city,area_name,property_type,deal_type,area_m2,rooms,baths,floors,sale_price_total,rent_price_month
0,Sanaa,Bayt-Meyad,apartment,rent,103.8,4,1,1,NaN,170543
1,Sanaa,Hayel,apartment,rent,132.3,3,2,1,NaN,160951
2,Sanaa,Hayel,apartment,rent,143.9,3,1,1,NaN,188453
3,Sanaa,Faj-Attan-Upper,apartment,rent,110.0,4,3,1,NaN,194989
4,Sanaa,Asr,apartment,rent,79.2,4,3,1,NaN,109613


In [4]:
# نختار فقط الصفوف اللي نوعها sale
sale_df = df[df["deal_type"] == "sale"].copy()

# Features (المدخلات)
X_sale = sale_df[["city", "area_name", "property_type", "deal_type",
                  "area_m2", "rooms", "baths", "floors"]]

# Target (المخرج / السعر)
y_sale = sale_df["sale_price_total"]

# الأعمدة الرقمية والتصنيفية
numeric_cols = ["area_m2", "rooms", "baths", "floors"]
cat_cols = ["city", "area_name", "property_type", "deal_type"]

# تجهيز الـ preprocessing: ترميز النصوص + تمرير الأرقام كما هي
preprocess_sale = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

# تعريف مودل XGBoost
sale_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# بناء Pipeline: preprocessing + model
sale_pipe = Pipeline(steps=[
    ("preprocess", preprocess_sale),
    ("model", sale_model)
])

# تقسيم Train / Test
X_train_sale, X_test_sale, y_train_sale, y_test_sale = train_test_split(
    X_sale, y_sale, test_size=0.2, random_state=42
)

# تدريب المودل
sale_pipe.fit(X_train_sale, y_train_sale)

# التنبؤ على بيانات الاختبار
y_pred_sale = sale_pipe.predict(X_test_sale)

# حساب متوسط الخطأ المطلق
mae_sale = mean_absolute_error(y_test_sale, y_pred_sale)
print("Sale Model MAE:", mae_sale)


Sale Model MAE: 6817972.5


In [5]:
joblib.dump(sale_pipe, "sale_model_xgb.pkl")


['sale_model_xgb.pkl']

In [15]:
# نختار فقط الصفوف اللي نوعها rent
rent_df = df[df["deal_type"] == "rent"].copy()

# Features
X_rent = rent_df[["city", "area_name", "property_type", "deal_type",
                  "area_m2", "rooms", "baths", "floors"]]

# Target
y_rent = rent_df["rent_price_month"]

numeric_cols_rent = ["area_m2", "rooms", "baths", "floors"]
cat_cols_rent = ["city", "area_name", "property_type", "deal_type"]

preprocess_rent = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_cols_rent),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_rent),
    ]
)

rent_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

rent_pipe = Pipeline(steps=[
    ("preprocess", preprocess_rent),
    ("model", rent_model)
])

X_train_rent, X_test_rent, y_train_rent, y_test_rent = train_test_split(
    X_rent, y_rent, test_size=0.2, random_state=42
)

rent_pipe.fit(X_train_rent, y_train_rent)

y_pred_rent = rent_pipe.predict(X_test_rent)
mae_rent = mean_absolute_error(y_test_rent, y_pred_rent)
print("Rent Model MAE:", mae_rent)


Rent Model MAE: 18621.802734375


In [16]:
joblib.dump(rent_pipe, "rent_model_xgb.pkl")


['rent_model_xgb.pkl']

In [24]:
import joblib
import pandas as pd

# تحميل المودل المحفوظ
sale_model = joblib.load("rent_model_xgb.pkl")

# مثال اختبار لعقار للبيع
test_sample = pd.DataFrame([{
    "city": "sanaa",
    "area_name": "Hadda",
    "property_type": "house",
    "deal_type": "rent",
    "area_m2": 150,
    "rooms": 2,
    "baths": 1,
    "floors": 1
}])

# التوقع
predicted_price = sale_model.predict(test_sample)[0]

print("Predicted Sale Price:", int(predicted_price))


Predicted Sale Price: 272607
